## Priprema okruženja

Instaliraju se dodatni paketi, uvoze potrebne biblioteke, postavlja se fiksni seed za reprodukovanost rezultata i bira se GPU (cuda) kao uređaj za izvršavanje.

Korišćenje GPU-a umesto CPU-a je izabrano zbog veće brzine treniranja.

Korišćen je T4.

In [ ]:
!pip install scikit-learn -q
!pip install onnxscript -q
import time
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
from google.colab import files

torch.manual_seed(42)

device = torch.device("cuda")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 18.7 MB/s eta 0:00:00


## Učitavanje i podela MNIST skupa

Preuzima se MNIST skup podataka, slike se normalizuju, trening skup se deli na trening i validacioni deo, i prave se DataLoader-i za trening, validaciju i test.

MNIST sadrži crno-bele slike rukom pisanih cifara dimenzija 28x28 piksela. 
Vrednosti za normalizaciju (0.1307 i 0.3081) odgovaraju srednjoj vrednosti i standardnoj devijaciji piksela celog trening skupa.

Izdvajanje posebnog validacionog skupa od 5000 primera omogućava praćenje performansi modela na podacima koje nije video tokom optimizacije, što je neophodno za rano zaustavljanje i biranje najboljih težina. Različite veličine batch-a za trening (64) i validaciju/test (256) su uobičajena praksa.

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_data, val_data = random_split(train_set, [55000, 5000])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=256, shuffle=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

print("train:", len(train_data))
print("val:", len(val_data))
print("test:", len(test_set))

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.02MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 129kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.6MB/s]

train: 55000
val: 5000
test: 10000


## Definicija LeNet5 arhitekture

Definiše se klasa LeNet5 sa konvolucionim, pooling i potpuno povezanim slojevima, i pravi se instanca modela na GPU-u.

Mreža naizmenično primenjuje konvolucione slojeve za izdvajanje prostornih obeležja i max pooling slojeve za smanjenje dimenzionalnosti i uvođenje otpornosti na male translacije u slici.

In [3]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


model = LeNet5().to(device)
print(model)

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
  (relu): ReLU()
)


## Funkcija greške i optimizator

Bira se unakrsna entropija kao funkcija greške i Adam kao optimizator za ažuriranje težina modela.

Vrednost 1e-3 za stopu je uobičajen podrazumevani izbor koji za većinu manjih mreža poput LeNet5 daje dobre rezultate bez dodatnog podešavanja.

In [4]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Trening modela

Model se trenira kroz više epoha, prati se greška na treningu i tačnost na validaciji, koristi se rano zaustavljanje i čuvaju se težine modela sa najboljom validacionom tačnošću.

U svakoj epohi model prvo prolazi kroz ceo trening skup u režimu treniranja, računajući grešku i ažurirajući težine propagacijom unazad, a zatim se prebacuje u režim evaluacije da bi se izmerila tačnost na validacionom skupu bez ažuriranja parametara. Rano zaustavljanje sa parametrom patience 3 znači da se treniranje prekida ukoliko validaciona tačnost ne poraste tri epohe zaredom.

Čuvanje težina samo kada se validaciona tačnost poboljša obezbeđuje da se na kraju koristi verzija modela koja najbolje generalizuje, a ne nužno ona iz poslednje epohe. Postavljanje maksimalnog broja epoha na 30 služi kao gornja granica koja se u praksi retko dostiže zahvaljujući ranom zaustavljanju, što se i vidi iz ispisa gde je treniranje stalo na 14. epohi.

In [5]:
max_epochs = 30
patience = 3

best_val_acc = 0
epochs_no_improve = 0

for epoch in range(max_epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            output = model(images)
            pred = output.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"epoch {epoch+1}/{max_epochs}  loss {avg_loss:.4f}  val_acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "lenet5_best.pth")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print("early stopping na epohi", epoch + 1)
        break

model.load_state_dict(torch.load("lenet5_best.pth"))

epoch 1/30  loss 0.2382  val_acc 0.9698
epoch 2/30  loss 0.0672  val_acc 0.9806
epoch 3/30  loss 0.0458  val_acc 0.9856
epoch 4/30  loss 0.0355  val_acc 0.9860
epoch 5/30  loss 0.0296  val_acc 0.9866
epoch 6/30  loss 0.0250  val_acc 0.9836
epoch 7/30  loss 0.0221  val_acc 0.9872
epoch 8/30  loss 0.0174  val_acc 0.9850
epoch 9/30  loss 0.0161  val_acc 0.9878
epoch 10/30  loss 0.0150  val_acc 0.9868
epoch 11/30  loss 0.0116  val_acc 0.9884
epoch 12/30  loss 0.0132  val_acc 0.9882
epoch 13/30  loss 0.0102  val_acc 0.9868
epoch 14/30  loss 0.0086  val_acc 0.9840
early stopping na epohi 14


<All keys matched successfully>

## Evaluacija na test skupu

Najbolji sačuvani model se testira na test skupu, računa se konačna tačnost i matrica konfuzije.

Test skup je potpuno odvojen od trening i validacionog skupa i model ga nije video ni tokom treniranja ni tokom biranja najbolje epohe, tako da tačnost izmerena na njemu daje realniju procenu ponašanja modela na potpuno novim podacima.

In [6]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        output = model(images)
        pred = output.argmax(dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test_acc {test_acc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
print(cm)

test_acc 0.9891
[[ 972    1    0    1    0    5    1    0    0    0]
 [   0 1130    1    3    0    1    0    0    0    0]
 [   4    0 1019    4    0    0    0    2    3    0]
 [   0    0    0 1009    0    1    0    0    0    0]
 [   0    0    0    0  978    0    0    0    0    4]
 [   0    0    0   11    0  880    1    0    0    0]
 [   1    2    0    0    0    5  949    0    1    0]
 [   0    0    8    7    0    0    0 1011    1    1]
 [   0    1    1    7    1    0    0    0  962    2]
 [   2    0    0    5   10    8    0    2    1  981]]


## Izvoz modela u ONNX format

Model se izvozi u ONNX format sa promenljivom batch dimenzijom, koristeći opset verziju 21.


In [9]:
model.eval()
dummy_input = torch.randn(1, 1, 28, 28).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "lenet5.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=21
)

print("model izvezen")

/tmp/ipykernel_893/2089683668.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `LeNet5([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LeNet5([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
model izvezen


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## Preuzimanje izvezenog modela

Izvezeni ONNX fajl se preuzima lokalno iz Colab okruženja.

Preuzeti lenet5.onnx fajl predstavlja polaznu tačku za sledeći korak, odnosno kvantizaciju modela i poređenje performansi i tačnosti kvantizovane verzije.

In [ ]:
files.download("lenet5.onnx")
files.download("lenet5.onnx.data")
files.download("lenet5_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>